
## 3. `calc_damping()` → damp_score_mult, damp_score_offset (with `last_supper()`) ⬜ 🟠 ❓ (VIII.10, VIII.11)

Notebook: [Testing/damping.ipynb](Testing/damping.ipynb)

**last_supper** = the last 3 cooked meals, ignored if cooked more than 4 days ago. Their ingredients come from `recipe_ingredients`.

Build **one** merged list from all last_supper meals (once, before looping the candidates):

| List | Content |
|---|---|
| `ls_origins` | all origins |
| `ls_methods` | all cooking methods |
| `ls_specs` | only sweet, soup, casserole (no other specs) |
| `ls_sides` | all side_dish ingredients |
| `ls_mains` | all main ingredients |
| `ls_meats` | all ingredients with `veg_type == meat` (each one counts for `len`) |

**Rules per candidate** – start with `damp_score_mult = 1`, `damp_score_offset = 0`:

| Condition | Effect |
|---|---|
| season overlap (current season, "all" also) | offset + 3 |
| candidate origin in `ls_origins` | offset − 1 |
| candidate cooking method in `ls_methods` | nothing |
| candidate sweet, soup or casserole in `ls_specs` | mult · (1 − 0.8) |
| candidate side_dish in `ls_sides` | offset − 3 |
| candidate is meat | mult · (1 − 0.1 · len(`ls_meats`)) — note: (1−x)ⁿ ≥ 1−nx |
| candidate meat in `ls_meats` | offset − 5 |
| main ingredient in `ls_mains` and it is meat or side_dish | mult · (1 − 0.1) |
| main ingredient in `ls_mains`, otherwise | offset − 3 |

→ every rule only checks "in list: yes / no" → fires at most once per candidate.

❓ Note: a negative offset might be too strong (VIII.10).

In [1]:
import pandas as pd
import numpy as np

today = pd.Timestamp.today().normalize()

cooked_meals = pd.DataFrame({
    "recipe_id": [1, 2, 3, 4],
    "cooked_at": today - pd.to_timedelta([1, 2, 6, 3], unit="D"),
})
recipes = pd.DataFrame({
    "id":             [1, 2, 3, 4, 5, 6, 7],
    "origin":         ["italian", "asian", "german", "italian", "asian", "french", "mexican"],
    "cooking_method": ["bake", "fry", "boil", "boil", "fry", "bake", "grill"],
    "spec":           ["casserole", None, "soup", None, "sweet", None, None],
    "season":         [["all"], ["autumn"], ["winter"], ["summer"], ["autumn", "winter"], ["spring"], ["all"]],
})
recipe_ingredients = pd.DataFrame({
    "recipe_id":  [1, 1, 2, 2, 4, 4, 5, 6, 6, 7, 7],
    "ingredient": ["pasta", "beef", "rice", "chicken", "potato", "pork", "apple", "pasta", "salmon", "rice", "beef"],
    "role":       ["side_dish", "main", "side_dish", "main", "side_dish", "main", "main", "side_dish", "main", "side_dish", "main"],
    "veg_type":   ["vegan", "meat", "vegan", "meat", "vegan", "meat", "vegan", "vegan", "fish", "vegan", "meat"],
})

In [2]:
SPECS = {"sweet", "soup", "casserole"}

def current_season(d):
    return {12: "winter", 1: "winter", 2: "winter",
            3: "spring", 4: "spring", 5: "spring",
            6: "summer", 7: "summer", 8: "summer"}.get(d.month, "autumn")

recent = cooked_meals[cooked_meals["cooked_at"] >= today - pd.Timedelta(days=4)]
last_supper = recent.sort_values("cooked_at", ascending=False)["recipe_id"].head(3).to_numpy()

ls_recipes = recipes[recipes["id"].isin(last_supper)]
ls_ingr    = recipe_ingredients[recipe_ingredients["recipe_id"].isin(last_supper)]

ls_origins = set(ls_recipes["origin"].dropna())
ls_methods = set(ls_recipes["cooking_method"].dropna())
ls_specs   = set(ls_recipes["spec"].dropna()) & SPECS
ls_sides   = set(ls_ingr.loc[ls_ingr["role"] == "side_dish", "ingredient"])
ls_mains   = set(ls_ingr.loc[ls_ingr["role"] == "main", "ingredient"])
ls_meats   = list(ls_ingr.loc[ls_ingr["veg_type"] == "meat", "ingredient"])   # Liste: len() zählt jedes Fleisch

print(last_supper, ls_origins, ls_methods, ls_specs, ls_sides, ls_mains, ls_meats, sep="\n")

[1 2 4]
{'asian', 'italian'}
{'boil', 'bake', 'fry'}
{'casserole'}
{'rice', 'potato', 'pasta'}
{'beef', 'chicken', 'pork'}
['beef', 'chicken', 'pork']


In [3]:
def damp_score(rec, ingr, season):
    mult, offset = 1.0, 0.0
    sides = set(ingr.loc[ingr["role"] == "side_dish", "ingredient"])
    mains = set(ingr.loc[ingr["role"] == "main", "ingredient"])
    meats = set(ingr.loc[ingr["veg_type"] == "meat", "ingredient"])

    if {season, "all"} & set(rec["season"]):   offset += 3
    if rec["origin"] in ls_origins:            offset -= 1
    if rec["cooking_method"] in ls_methods:    pass
    if rec["spec"] in ls_specs:                mult *= 1 - 0.8
    if sides & ls_sides:                       offset -= 3
    if meats:                                  mult *= max(0, 1 - 0.1 * len(ls_meats))
    if meats & set(ls_meats):                  offset -= 5

    repeated = mains & ls_mains
    if repeated:
        if repeated & (meats | set(ls_meats) | ls_sides):
            mult *= 1 - 0.1
        else:
            offset -= 3
    return mult, offset

In [4]:
season = current_season(today)
candidates = recipes[~recipes["id"].isin(last_supper)]

rows = []
for _, rec in candidates.iterrows():
    ingr = recipe_ingredients[recipe_ingredients["recipe_id"] == rec["id"]]
    damp_score_mult, damp_score_offset = damp_score(rec, ingr, season)
    rows.append({"id": rec["id"],
                 "damp_score_mult": damp_score_mult,
                 "damp_score_offset": damp_score_offset})

scores = pd.DataFrame(rows)
scores

,id,damp_score_mult,damp_score_offset
0,3,1.00,0.0
1,5,1.00,2.0
2,6,1.00,-3.0
3,7,0.63,-5.0


## Sanity Checks

Expected behavior. 🟠 = not tested yet; "old" = value from the old test script, re-check here.

| Situation | Expected | Status |
|---|---|---|
| same meat + side + main as yesterday | strongly damped, but still positive | 🟠 old: 3.72 at base 12 |
| other meat type than last_supper | only ×(1 − 0.1·n), no −5 | 🟠 |
| same main, neither meat nor side | −3 | 🟠 |
| soup after soup | strongly damped | 🟠 old: ×0.2 |
| nothing in common with last_supper | unchanged (+ season) | 🟠 |
| every damping rule | fires max. once | 🟠 |